# Get answers from LLMs

Pass questions to LLMs with no extra information and store the answers.

In [175]:
import json
from pathlib import Path
from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.settings import ModelSettings
import nest_asyncio
nest_asyncio.apply() # To be able to run pydantic in jupyter notebooks preventing event loop issues


load_dotenv()

BASE_PATH = Path.cwd()
# DATA_FILE_PATH = BASE_PATH.joinpath("20251012_llm_answers_baseline_from_20250804_101435_generated_questions.json")
DATA_FILE_PATH = BASE_PATH.joinpath("20251012_llm_answers_baseline_from_20250804_101435_llm_base_answers.json") # To skip already generated answers
OUTPUT_FILE_PATH = BASE_PATH.joinpath("20251012_llm_answers_baseline_from_20250804_101435_llm_base_answers.json")
with open(DATA_FILE_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

def generate_llm_responses(question, model, system_prompt, model_settings, tools=None):
    agent = Agent(
        model=model,
        system_prompt=system_prompt,
        model_settings=model_settings
        )
    agent.model_settings

    result = agent.run_sync(
        question,
        model_settings=model_settings,
    )

    respuesta = {
        "texto": result.output,
        "metadata": {# response metadata
            "model_settings": agent.model_settings,
            "model_version": result.response.model_name,
            "provider_name": result.response.provider_name,
            "timestamp": result.timestamp().isoformat(),
            "usage": {
                "tokens": {
                    "input_tokens": result.usage().input_tokens,
                    "output_tokens": result.usage().output_tokens,
                    "tokens_details": result.usage().details,
                },
                "cost": {
                    "input_price": float(result.response.cost().input_price),
                    "output_price": float(result.response.cost().output_price),
                    "total_price": float(result.response.cost().total_price),
                    "model_input_price_mtok": float(result.response.cost().model_price.input_mtok),
                    "model_output_price_mtok": float(result.response.cost().model_price.output_mtok),
                    "model_id": result.response.cost().model.id,
                    "model_name": result.response.cost().model.name,
                },
            },
        }
    }
    print(result)

    return respuesta

In [ ]:
tools = []

model_settings = ModelSettings(
    temperature=0.0, # As deterministic responses as possible
    # top_p=0.1,
)

system_prompt = """
Eres un asistente legal experto en derecho Argentino. Respondes preguntas legales basadas en la Constitución Nacional Argentina, Códigos, Leyes y jurisprudencia. Siempre que sea posible citas las fuentes legales en tu respuesta. Si no sabes la respuesta, dices que no lo sabes.
Dentro del texto de la respuesta debe mencionarse de donde provino la afirmación. Por ejemplo "según el artículo... de la ...". Las respuestas deben ser claras, concisas y directas al punto.
Solo retorna la respuesta.
"""

models=[
    "openai:gpt-5-nano",
    "openai:gpt-5-mini",
    "openai:gpt-5",
]


for pregunta in data:
    for model in models:
        # Check if response for this model already exists
        # This checks does not account for different system prompts
        if any(
                model.endswith(resp["metadata"]["usage"]["cost"]["model_id"])
                and resp["metadata"]["model_settings"] == model_settings
                for resp in pregunta.get("llm_responses_no_rag", [])
            ):
            print(f"Response for model '{model}', question '{pregunta['numero_pregunta']}', subpregunta '{pregunta.get('numero_subpregunta', None)}', and model settings already exists. Skipping...")
            continue

        if "llm_responses_no_rag" not in pregunta:
            pregunta["llm_responses_no_rag"] = []

        print(f"Generating response with model '{model}', question '{pregunta['numero_pregunta']}' and subpregunta '{pregunta.get('numero_subpregunta', None)}'")
        pregunta["llm_responses_no_rag"].append(generate_llm_responses(pregunta["texto"], model, system_prompt, model_settings))

        with open(OUTPUT_FILE_PATH, encoding="utf-8", mode="w") as f:
            f.write(json.dumps(data, ensure_ascii=False, indent=4))

Response for model 'openai:gpt-5-nano', question '1', subpregunta '1', and model settings already exists. Skipping...
Response for model 'openai:gpt-5-mini', question '1', subpregunta '1', and model settings already exists. Skipping...
Response for model 'openai:gpt-5', question '1', subpregunta '1', and model settings already exists. Skipping...
Response for model 'openai:gpt-5-nano', question '1', subpregunta '2', and model settings already exists. Skipping...
Response for model 'openai:gpt-5-mini', question '1', subpregunta '2', and model settings already exists. Skipping...
Response for model 'openai:gpt-5', question '1', subpregunta '2', and model settings already exists. Skipping...
Response for model 'openai:gpt-5-nano', question '1', subpregunta '3', and model settings already exists. Skipping...
Response for model 'openai:gpt-5-mini', question '1', subpregunta '3', and model settings already exists. Skipping...
Response for model 'openai:gpt-5', question '1', subpregunta '3', 

In [177]:
pregunta

{'numero_pregunta': 25,
 'numero_subpregunta': None,
 'texto': 'Una empresa de explotación de litio consulta sobre el régimen de multiplicidad de pertenencias aplicable a yacimientos de litio en 2025. ¿Cuál es el régimen de asignación/multiplicidad de pertenencias para yacimientos de litio al año 2025?',
 'contexto_para_respuesta': ['Artículo 351'],
 'respuesta': 'Para yacimientos de litio (considerados de tipo diseminado de la primera categoría por el Artículo 76), el número de pertenencias asignadas a descubridores y compañías se multiplica por cinco (5), según lo establece el Artículo 351.',
 'llm_responses': [{'texto': 'No existe, a 2025, un régimen federal único de “multiplicidad de pertenencias” aplicable a yacimientos de litio. El régimen vigente se regula por la legislación minera de cada provincia y por la Constitución Nacional. En particular:\n\n- Constitución Nacional, artículo 124: las riquezas del subsuelo pertenecen a las provincias (la Provincia regula la extracción y ti